In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Load the real-world UCI Appliances Energy Dataset
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"
print("Downloading real-world dataset from UCI Repository...")
df = pd.read_csv(data_url)

print(f"Dataset successfully loaded. Total rows: {df.shape[0]}, Total columns: {df.shape[1]}")
display(df.head(3))

Dataset successfully loaded. Total rows: 19735, Total columns: 29


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668


In [2]:
# Convert date column to datetime object
df['date'] = pd.to_datetime(df['date'])

# Extract cyclical time components
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Drop the raw date string and non-predictive target columns to isolate features
# We are predicting 'Appliances' energy consumption (Wh) as a proxy for grid workload demand
X = df.drop(cols := ['date', 'Appliances', 'lights'], axis=1, errors='ignore')
y = df['Appliances']

print("Features selected for the model:")
print(list(X.columns))

Features selected for the model:
['T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2', 'hour', 'day_of_week', 'month']


In [3]:
# Perform an 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Matrix Shape: {X_train.shape}")
print(f"Testing Matrix Shape: {X_test.shape}")

Training Matrix Shape: (15788, 29)
Testing Matrix Shape: (3947, 29)


In [4]:
print("Initializing and training Random Forest Regressor...")

# Define the ensemble model
model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

# Fit the model to our real-world training data
model.fit(X_train, y_train)

print("Model training complete.")

Initializing and training Random Forest Regressor...
Model training complete.


In [5]:
# Generate predictions on unseen test features
y_pred = model.predict(X_test)

# Calculate metrics
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Real-World Model Performance Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} Wh")
print(f"R² Score (Variance Explained): {r2 * 100:.2f}%")

--- Real-World Model Performance Evaluation ---
Mean Absolute Error (MAE): 33.46 Wh
R² Score (Variance Explained): 53.73%


In [6]:
# Export the model binary and feature schema list
joblib.dump(model, 'carbon_model.pkl')
joblib.dump(list(X.columns), 'model_features.pkl')

print("Artifacts generated successfully:")
print("1. carbon_model.pkl (Random Forest Binary)")
print("2. model_features.pkl (Feature Schema List)")

Artifacts generated successfully:
1. carbon_model.pkl (Random Forest Binary)
2. model_features.pkl (Feature Schema List)
